<a href="https://colab.research.google.com/github/mohamedelguendy/vigenere-cipher-cracker/blob/main/vigenere-cipher-cracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### recover the key and plaintext without knowing the key

In [ ]:
import string
from collections import Counter

ALPHABET = string.ascii_uppercase
ENGLISH_FREQ = {
    'A': 0.08167, 'B': 0.01492, 'C': 0.02782, 'D': 0.04253,
    'E': 0.12702, 'F': 0.02228, 'G': 0.02015, 'H': 0.06094,
    'I': 0.06966, 'J': 0.00153, 'K': 0.00772, 'L': 0.04025,
    'M': 0.02406, 'N': 0.06749, 'O': 0.07507, 'P': 0.01929,
    'Q': 0.00095, 'R': 0.05987, 'S': 0.06327, 'T': 0.09056,
    'U': 0.02758, 'V': 0.00978, 'W': 0.02360, 'X': 0.00150,
    'Y': 0.01974, 'Z': 0.00074
}

def clean_text(text):
    return ''.join([c for c in text.upper() if c in ALPHABET])

# -------------------------------
# Step 1: Index of Coincidence
# -------------------------------
def index_of_coincidence(text):
    N = len(text)
    freqs = Counter(text)
    ic = sum(f * (f - 1) for f in freqs.values()) / (N * (N - 1)) if N > 1 else 0
    return ic

def estimate_key_lengths(ciphertext, max_len=12):
    scores = []
    for key_len in range(1, max_len + 1):
        ic_values = []
        for i in range(key_len):
            segment = ciphertext[i::key_len]
            ic_values.append(index_of_coincidence(segment))
        avg_ic = sum(ic_values) / len(ic_values)
        scores.append((key_len, avg_ic))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:5]  # top 5 candidates

# -------------------------------
# Step 2: Frequency Analysis
# -------------------------------
def chi_squared(text):
    N = len(text)
    freqs = Counter(text)
    chi = 0
    for letter in ALPHABET:
        observed = freqs.get(letter, 0)
        expected = ENGLISH_FREQ[letter] * N
        chi += (observed - expected) ** 2 / expected if expected > 0 else 0
    return chi

def find_key(ciphertext, key_len):
    key = ""

    for i in range(key_len):
        segment = ciphertext[i::key_len]
        best_shift = None
        best_score = float('inf')

        for shift in range(26):
            decrypted = ""
            for c in segment:
                new_char = chr((ord(c) - ord('A') - shift) % 26 + ord('A'))
                decrypted += new_char

            score = chi_squared(decrypted)

            if score < best_score:
                best_score = score
                best_shift = shift

        key += chr(best_shift + ord('A'))

    return key

# -------------------------------
# Step 3: Decrypt
# -------------------------------
def decrypt_vigenere(ciphertext, key):
    plaintext = ""
    key_len = len(key)

    for i, c in enumerate(ciphertext):
        shift = ord(key[i % key_len]) - ord('A')
        new_char = chr((ord(c) - ord('A') - shift) % 26 + ord('A'))
        plaintext += new_char

    return plaintext

# -------------------------------
# Main Crack Function
# -------------------------------
def crack_vigenere(ciphertext):
    ciphertext = clean_text(ciphertext)

    print("Estimating key lengths...")
    key_lengths = estimate_key_lengths(ciphertext)

    results = []

    for key_len, _ in key_lengths:
        key = find_key(ciphertext, key_len)
        plaintext = decrypt_vigenere(ciphertext, key)
        score = chi_squared(plaintext)

        results.append((key, plaintext, score))

    results.sort(key=lambda x: x[2])

    best_key, best_plaintext, _ = results[0]

    return best_key, best_plaintext

# -------------------------------
# Example Usage
# -------------------------------
if __name__ == "__main__":
    cipher = input("Enter ciphertext: ")

    key, plaintext = crack_vigenere(cipher)

    print("\nRecovered Key:", key)
    print("Decrypted Text:\n", plaintext)

Enter ciphertext: dawdfeawd
Estimating key lengths...

Recovered Key: MAWD
Decrypted Text:
 RAAATEETR
